In [4]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# Reynolds number
REYNOLDS_NUMBER = 100.0

# lambda_BD = Boundary loss scale factor
BOUNDARY_SCALE = 1.0

In [6]:
class LidCavityPINN(nn.Module):
    def __init__(self):
        super(LidCavityPINN, self).__init__()

        HIDDEN_LAYER_COUNT = 8
        NEURON_DENSITY = 20
        INPUT_DIM = 3
        OUTPUT_DIM = 3

        # activation = torch.nn.Tanh()
        activation = torch.nn.SiLU()
        # activation = SinActivation()

        self.hidden_layers = nn.Sequential(
            nn.Linear(INPUT_DIM, NEURON_DENSITY),
            activation,
        )
        for _ in range(HIDDEN_LAYER_COUNT):
            self.hidden_layers.append(nn.Linear(NEURON_DENSITY, NEURON_DENSITY))
            self.hidden_layers.append(activation)

        self.hidden_layers.append(nn.Linear(NEURON_DENSITY, OUTPUT_DIM))

    def forward(self, x, y, t):
        return self.hidden_layers(torch.cat([x, y, t], dim=1))


model = LidCavityPINN().to(device)

In [ ]:
from itertools import product

BOUNDING_SIDE    = 50

# rand automatically generates values in uniform distribution in the range [0, 1)

input_domain = torch.tensor(list(product(x, y)), device=device, requires_grad=True)

NameError: name 'COLLOC_POINTS' is not defined

In [ ]:
lid_boundary_mask = input_domain[:, 1] == 1.0
lid_boundary_colloc = input_domain[lid_boundary_mask]

no_slip_boundary_mask = (
      (input_domain[:, 0] == 0.0)
    | (input_domain[:, 0] == 1.0)
    | ((input_domain[:, 1] == 0.0) & (input_domain[:, 1] != 1.0))
)
no_slip_boundary_colloc = input_domain[no_slip_boundary_mask]

boundary_mask = no_slip_boundary_mask | lid_boundary_mask

interior_colloc = input_domain[~boundary_mask]


In [ ]:
import matplotlib.pyplot as plt

plt.scatter(no_slip_boundary_colloc[:,0],no_slip_boundary_colloc[:,1], c="r")
plt.scatter(lid_boundary_colloc[:,0],lid_boundary_colloc[:,1], c="blue")
plt.scatter(interior_colloc[:,0],interior_colloc[:,1], c="black")
plt.show()

In [ ]:
def convert_to_device(*args):
    return [arg.clone().detach().requires_grad_(True).to(device) for arg in args]


input_domain, no_slip_boundary_colloc, lid_boundary_colloc, interior_colloc = (
    convert_to_device(
        input_domain, no_slip_boundary_colloc, lid_boundary_colloc, interior_colloc
    )
)

In [ ]:
def no_slip_loss_fnc():
    _, u, v = model(no_slip_boundary_colloc)
    return torch.mean(u**2 + v**2)


def lid_driven_loss_fnc():
    _, u, v = model(lid_boundary_colloc)
    return torch.mean(v**2 + (u - torch.ones_like(u)) ** 2)


def pde_loss_fnc():
    p, u, v = model(interior_colloc)

    grad_out = torch.ones_like(u)
    dp_d = torch.autograd.grad(p, interior_colloc, grad_outputs=grad_out, create_graph=True)[0]
    dp_dx = dp_d[:, 0]
    dp_dy = dp_d[:, 1]

    du_d = torch.autograd.grad(u, interior_colloc, grad_outputs=grad_out, create_graph=True)[0]
    du_dx = du_d[:, 0]
    du_dy = du_d[:, 1]

    dv_d = torch.autograd.grad(v, interior_colloc, grad_outputs=grad_out, create_graph=True)[0]
    dv_dx = dv_d[:, 0]
    dv_dy = dv_d[:, 1]

    du_dxd = torch.autograd.grad(du_dx, interior_colloc, grad_outputs=grad_out, create_graph=True)[0]
    du_dxdx = du_dxd[:, 0]
    du_dyd = torch.autograd.grad(du_dy, interior_colloc, grad_outputs=grad_out, create_graph=True)[0]
    du_dydy = du_dyd[:, 1]

    dv_dxd = torch.autograd.grad(dv_dx, interior_colloc, grad_outputs=grad_out, create_graph=True)[0]
    dv_dxdx = dv_dxd[:, 0]
    dv_dyd = torch.autograd.grad(dv_dy, interior_colloc, grad_outputs=grad_out, create_graph=True)[0]
    dv_dydy = dv_dyd[:, 1]

    res_mom_x = u * du_dx + v * du_dy + dp_dx - 1 / REYNOLDS_NUMBER * (du_dxdx + du_dydy)
    res_mom_y = u * dv_dx + v * dv_dy + dp_dy - 1 / REYNOLDS_NUMBER * (dv_dxdx + dv_dydy)

    return torch.mean(res_mom_x**2 + res_mom_y**2)


In [ ]:
def loss_fnc():
    no_slip_loss = no_slip_loss_fnc()
    lid_driven_loss = lid_driven_loss_fnc()
    pde_loss = pde_loss_fnc()

    return BOUNDARY_SCALE * (no_slip_loss + lid_driven_loss) + pde_loss

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)

def training_step():
    optimizer.zero_grad()
    loss = loss_fnc()
    loss.backward()
    return loss

In [ ]:
import datetime

EPOCHS = 100

loss_hist = []

start = datetime.datetime.now()
for i in range(EPOCHS):
    optimizer.zero_grad()
    

    loss_hist.append(loss_fnc().detach().cpu().numpy())

    if i % 100 == 0:
        print(f"Iteration: {i} | Loss: {loss_hist[-1]:.8e}")
        print(f"\t Iteration duration: {(datetime.datetime.now()-start).total_seconds()}s")
        start = datetime.datetime.now()


In [ ]:
nx, ny = 50, 50
x = torch.linspace(0, 1, nx)
y = torch.linspace(0, 1, ny)
X, Y = torch.meshgrid(x, y)

XY = torch.cat([X.reshape(-1, 1), Y.reshape(-1, 1)], dim=1)
_, u_d, v_d = model(input_domain)
u_d = u_d.detach().cpu().numpy().reshape(nx, ny).numpy()
v_d = v_d.detach().cpu().numpy().reshape(nx, ny).numpy()

plt.figure(figsize=(8, 6))
plt.streamplot(
    X.numpy(), Y.numpy(),
    u_d, v_d, density=2, linewidth=1, arrowsize=1
)
plt.title('Predicted Velocity Field (Streamlines)')
plt.xlabel('x')
plt.ylabel('y')
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.show()